# 07 - Evaluate the LoRA meta-model on the test set

Loads `artifacts/meta_jsonl/test.jsonl`, generates predictions with the LoRA-adapted Gemma-2-9B-it, and reports accuracy / macro-F1 / per-class F1 / confusion matrix.

Also computes three baselines on the same test set for honest comparison:

1. weighted-average argmax (no LLM)
2. best single base model (by val accuracy)
3. zero-shot Gemma-2-9B-it on the same prompts (no LoRA)

In [ ]:
%pip install -q 'transformers>=4.44' 'peft>=0.11' 'bitsandbytes>=0.43' accelerate datasets scikit-learn matplotlib seaborn

In [ ]:
import os, sys, json, re

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

import numpy as np, pandas as pd
from tm_research.ensemble.utils_io import (
    META_JSONL_DIR, LORA_DIR, ARTIFACTS_DIR, METRICS_DIR,
    load_label_map, load_probs
)
from tm_research.ensemble.utils_stacking import (
    BASE_MODEL_NAMES, weighted_average, parse_label_from_completion
)
label_map = load_label_map()
with open(ARTIFACTS_DIR / 'weights.json', 'r', encoding='utf-8') as f:
    weights_payload = json.load(f)
weights = weights_payload['weights']
val_acc = weights_payload['val_accuracy_per_model']
print('weights', weights)
print('val accuracies', val_acc)

In [ ]:
test_rows = []
with open(META_JSONL_DIR / 'test.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        test_rows.append(json.loads(line))
y_test = np.array([label_map.label2id[r['gold']] for r in test_rows])
print('test size:', len(test_rows))

## Token length & training truncation audit

Counts Gemma tokens on `test.jsonl` (and `train.jsonl` for the SFT path) using the same chat template as inference (07) and training (06). Reports how often `[/WEIGHTED_AVG]`, `[/TEXT]`, or the `[TEXT]` body tail would be lost at `max_seq_len` from `lora_train_meta.json` (default 1024) vs inference `max_length=2048`.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoTokenizer
from tm_research.ensemble.utils_stacking import build_meta_system_prompt

MODEL_NAME = 'google/gemma-2-9b-it'
_lora_meta_path = ARTIFACTS_DIR / 'lora_train_meta.json'
if _lora_meta_path.is_file():
    with open(_lora_meta_path, 'r', encoding='utf-8') as _f:
        _lora_meta = json.load(_f)
    SYSTEM_PROMPT_AUDIT = _lora_meta.get('system_prompt') or weights_payload.get('system_prompt') or build_meta_system_prompt(label_map, weights)
    TRAIN_MAX_LEN = int(_lora_meta.get('max_seq_len', 800))
else:
    SYSTEM_PROMPT_AUDIT = weights_payload.get('system_prompt') or build_meta_system_prompt(label_map, weights)
    TRAIN_MAX_LEN = 800
INFER_MAX_LEN = 2048

_tok_src = str(LORA_DIR) if (LORA_DIR / 'tokenizer_config.json').is_file() else MODEL_NAME
_tok_kwargs = {}
if _tok_src == MODEL_NAME:
    if not os.environ.get('HF_TOKEN'):
        raise RuntimeError('Set HF_TOKEN to tokenize with the base Gemma tokenizer, or save the adapter tokenizer under artifacts/lora_adapter/.')
    _tok_kwargs['token'] = os.environ['HF_TOKEN']
audit_tok = AutoTokenizer.from_pretrained(_tok_src, **_tok_kwargs)
if audit_tok.pad_token is None:
    audit_tok.pad_token = audit_tok.eos_token


def _user_content(prompt_block: str) -> str:
    return SYSTEM_PROMPT_AUDIT + '\n\n' + prompt_block


def build_infer_chat(prompt_block: str) -> str:
    return audit_tok.apply_chat_template(
        [{'role': 'user', 'content': _user_content(prompt_block)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def build_train_chat(prompt_block: str, completion: str) -> str:
    return audit_tok.apply_chat_template(
        [
            {'role': 'user', 'content': _user_content(prompt_block)},
            {'role': 'assistant', 'content': completion},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )


def _truncate_decode(text: str, max_len: int) -> tuple[list[int], str]:
    enc = audit_tok(
        text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_len,
    )
    ids = enc['input_ids']
    return ids, audit_tok.decode(ids, skip_special_tokens=False)


def _prompt_tail_flags(prompt_block: str, decoded: str) -> dict:
    """Flags on the structured prompt region (after system prompt in user turn)."""
    # Decoded chat still contains user payload; marker checks are on full decode.
    text_open = '[TEXT]' in decoded
    text_close = '[/TEXT]' in decoded
    stack_close = '[/STACK]' in decoded
    weighted_close = '[/WEIGHTED_AVG]' in decoded
    orig_text = prompt_block.split('[TEXT]', 1)[-1].split('[/TEXT]', 1)[0].strip() if '[TEXT]' in prompt_block else ''
    in_decoded = orig_text in decoded if orig_text else True
    text_tail_cut = bool(orig_text) and not in_decoded
    return {
        'text_open': text_open,
        'text_close': text_close,
        'stack_close': stack_close,
        'weighted_avg_close': weighted_close,
        'text_tail_cut': text_tail_cut,
    }


def audit_rows(rows, split_name: str, *, train_mode: bool) -> pd.DataFrame:
    records = []
    for row in rows:
        prompt_block = row['prompt']
        if train_mode:
            completion = row.get('completion') or ''
            chat = build_train_chat(prompt_block, completion)
        else:
            chat = build_infer_chat(prompt_block)
        full_ids = audit_tok(chat, add_special_tokens=False)['input_ids']
        n_full = len(full_ids)
        trunc_ids, trunc_dec = _truncate_decode(chat, TRAIN_MAX_LEN)
        n_trunc = len(trunc_ids)
        flags_full = _prompt_tail_flags(prompt_block, chat)
        flags_trunc = _prompt_tail_flags(prompt_block, trunc_dec)
        records.append({
            'split': split_name,
            'idx': row.get('idx'),
            'mode': 'sft_train' if train_mode else 'infer',
            'n_tokens_full': n_full,
            'n_tokens_at_train_max': n_trunc,
            'truncated_at_train_max': n_full > TRAIN_MAX_LEN,
            'chars_prompt_block': len(prompt_block),
            'chars_text_field': len(row.get('text') or ''),
            'missing_weighted_avg_at_train_max': not flags_trunc['weighted_avg_close'],
            'missing_text_close_at_train_max': not flags_trunc['text_close'],
            'missing_stack_close_at_train_max': not flags_trunc['stack_close'],
            'text_tail_cut_at_train_max': flags_trunc['text_tail_cut'],
            'full_has_weighted_avg': flags_full['weighted_avg_close'],
        })
    return pd.DataFrame(records)


def _summarize(df: pd.DataFrame, title: str) -> None:
    n = len(df)
    if n == 0:
        print(title, '(empty)')
        return
    print(f'\n=== {title} (n={n}) ===')
    for col in ('n_tokens_full', 'n_tokens_at_train_max', 'chars_prompt_block', 'chars_text_field'):
        s = df[col]
        print(
            f'  {col}: min={int(s.min())} p50={int(s.median())} p95={int(s.quantile(0.95))} '
            f'max={int(s.max())} mean={s.mean():.1f}'
        )
    trunc = df['truncated_at_train_max']
    print(f'  truncated at train max ({TRAIN_MAX_LEN}): {int(trunc.sum())} ({100 * trunc.mean():.2f}%)')
    for flag in (
        'missing_weighted_avg_at_train_max',
        'missing_text_close_at_train_max',
        'missing_stack_close_at_train_max',
        'text_tail_cut_at_train_max',
    ):
        c = int(df[flag].sum())
        print(f'  {flag}: {c} ({100 * c / n:.2f}%)')
    over = df[df['truncated_at_train_max']].sort_values('n_tokens_full', ascending=False).head(5)
    if len(over):
        print('  top truncated idx (by n_tokens_full):', over['idx'].tolist())


# --- test.jsonl: inference path (same as generate_labels) ---
test_audit = audit_rows(test_rows, 'test', train_mode=False)
_summarize(test_audit, f'test.jsonl — inference chat (add_generation_prompt=True)')

# --- train.jsonl: SFT path (user + assistant, max_length during notebook 06) ---
train_rows_audit = []
with open(META_JSONL_DIR / 'train.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        train_rows_audit.append(json.loads(line))
train_audit = audit_rows(train_rows_audit, 'train', train_mode=True)
_summarize(train_audit, f'train.jsonl — SFT chat (user + assistant)')

# --- inference at eval max_length (2048) on test ---
infer_at_eval = []
for row in test_rows:
    chat = build_infer_chat(row['prompt'])
    full_ids = audit_tok(chat, add_special_tokens=False)['input_ids']
    _, trunc_dec = _truncate_decode(chat, INFER_MAX_LEN)
    flags = _prompt_tail_flags(row['prompt'], trunc_dec)
    infer_at_eval.append({
        'idx': row.get('idx'),
        'n_tokens_full': len(full_ids),
        'truncated_at_infer_max': len(full_ids) > INFER_MAX_LEN,
        'missing_weighted_avg_at_infer_max': not flags['weighted_avg_close'],
        'text_tail_cut_at_infer_max': flags['text_tail_cut'],
    })
infer_eval_df = pd.DataFrame(infer_at_eval)
print(f'\n=== test.jsonl — at inference max_length={INFER_MAX_LEN} ===')
print(
    f'  truncated: {int(infer_eval_df["truncated_at_infer_max"].sum())} '
    f'({100 * infer_eval_df["truncated_at_infer_max"].mean():.2f}%)'
)
print(
    f'  missing [/WEIGHTED_AVG]: {int(infer_eval_df["missing_weighted_avg_at_infer_max"].sum())} '
    f'({100 * infer_eval_df["missing_weighted_avg_at_infer_max"].mean():.2f}%)'
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(test_audit['n_tokens_full'], bins=40, alpha=0.85, label='test infer full')
axes[0].axvline(TRAIN_MAX_LEN, color='red', ls='--', label=f'train max={TRAIN_MAX_LEN}')
axes[0].axvline(INFER_MAX_LEN, color='green', ls='--', label=f'infer max={INFER_MAX_LEN}')
axes[0].set_xlabel('tokens'); axes[0].set_ylabel('count'); axes[0].set_title('test.jsonl inference tokens')
axes[0].legend()
axes[1].hist(train_audit['n_tokens_full'], bins=40, alpha=0.85, color='orange', label='train SFT full')
axes[1].axvline(TRAIN_MAX_LEN, color='red', ls='--', label=f'train max={TRAIN_MAX_LEN}')
axes[1].set_xlabel('tokens'); axes[1].set_title('train.jsonl SFT tokens')
axes[1].legend()
plt.tight_layout()
plt.show()

audit_out = METRICS_DIR / 'token_length_audit.json'
summary_payload = {
    'train_max_len': TRAIN_MAX_LEN,
    'infer_max_len': INFER_MAX_LEN,
    'tokenizer_source': _tok_src,
    'test_infer': {
        'n': int(len(test_audit)),
        'truncated_at_train_max': int(test_audit['truncated_at_train_max'].sum()),
        'missing_weighted_avg_at_train_max': int(test_audit['missing_weighted_avg_at_train_max'].sum()),
        'missing_text_close_at_train_max': int(test_audit['missing_text_close_at_train_max'].sum()),
        'text_tail_cut_at_train_max': int(test_audit['text_tail_cut_at_train_max'].sum()),
        'n_tokens_full': test_audit['n_tokens_full'].describe().to_dict(),
    },
    'train_sft': {
        'n': int(len(train_audit)),
        'truncated_at_train_max': int(train_audit['truncated_at_train_max'].sum()),
        'missing_weighted_avg_at_train_max': int(train_audit['missing_weighted_avg_at_train_max'].sum()),
        'missing_text_close_at_train_max': int(train_audit['missing_text_close_at_train_max'].sum()),
        'text_tail_cut_at_train_max': int(train_audit['text_tail_cut_at_train_max'].sum()),
        'n_tokens_full': train_audit['n_tokens_full'].describe().to_dict(),
    },
    'test_at_infer_max': {
        'truncated': int(infer_eval_df['truncated_at_infer_max'].sum()),
        'missing_weighted_avg': int(infer_eval_df['missing_weighted_avg_at_infer_max'].sum()),
    },
}
with open(audit_out, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)
print('saved', audit_out)
display(test_audit.head(3))

## Baselines

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

test_probs = {m: load_probs(m, 'test') for m in BASE_MODEL_NAMES}

def report(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\n=== {name} ===\n  acc={acc:.4f}  macro_f1={macro_f1:.4f}  micro_f1={micro_f1:.4f}')
    print(classification_report(
        y_true, y_pred,
        target_names=label_map.class_names,
        zero_division=0,
    ))
    return {'name': name, 'accuracy': acc, 'macro_f1': macro_f1, 'micro_f1': micro_f1}

results = []
weighted_test = weighted_average(test_probs, weights)
results.append(report('baseline_weighted_avg_argmax', y_test, weighted_test.argmax(1)))
best_base = max(val_acc, key=val_acc.get)
results.append(report(f'baseline_best_single_{best_base}', y_test, test_probs[best_base].argmax(1)))

## LoRA meta-model generation

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from tm_research.ensemble.utils_stacking import build_meta_system_prompt

MODEL_NAME = 'google/gemma-2-9b-it'
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('Set HF_TOKEN with Gemma-2 license accepted.')

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ['HF_TOKEN'])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto',
    torch_dtype=torch.bfloat16, attn_implementation='eager',
    token=os.environ['HF_TOKEN'],
)
lora_meta_path = ARTIFACTS_DIR / 'lora_train_meta.json'
with open(lora_meta_path, 'r', encoding='utf-8') as f:
    lora_meta = json.load(f)
# Load system prompt from lora_train_meta.json; fall back to weights.json or build fresh.
if lora_meta.get('system_prompt'):
    SYSTEM_PROMPT = lora_meta['system_prompt']
elif weights_payload.get('system_prompt'):
    SYSTEM_PROMPT = weights_payload['system_prompt']
else:
    SYSTEM_PROMPT = build_meta_system_prompt(label_map, weights)
# max_length for LoRA inference matches the training cap for a fair comparison.
LORA_INFER_MAX_LEN = int(lora_meta.get('max_seq_len', 800))
print('SYSTEM_PROMPT:', SYSTEM_PROMPT[:120], '...')
print('LORA_INFER_MAX_LEN:', LORA_INFER_MAX_LEN)
model_lora = PeftModel.from_pretrained(base, str(LORA_DIR))
model_lora.eval()

In [ ]:
def build_chat(prompt_block):
    return tok.apply_chat_template(
        [{'role': 'user', 'content': SYSTEM_PROMPT + '\n\n' + prompt_block}],
        tokenize=False, add_generation_prompt=True,
    )

@torch.inference_mode()
def generate_labels(model, rows, batch_size=4, max_new_tokens=12, max_length=None):
    """Generate label predictions for a list of meta-dataset rows.

    ``max_length`` controls tokenizer truncation. Pass ``LORA_INFER_MAX_LEN``
    for the LoRA model (matches training cap) and a larger value (e.g. 2048)
    for zero-shot Gemma so it sees the full prompt.
    """
    if max_length is None:
        max_length = LORA_INFER_MAX_LEN
    preds = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        chats = [build_chat(r['prompt']) for r in batch]
        enc = tok(chats, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(model.device)
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0,
            pad_token_id=tok.pad_token_id,
        )
        gen = out[:, enc['input_ids'].shape[1]:]
        decoded = tok.batch_decode(gen, skip_special_tokens=True)
        for txt in decoded:
            lab = parse_label_from_completion(txt, label_map)
            preds.append(lab)
        if i % (batch_size * 10) == 0:
            print(f'  {i}/{len(rows)}')
    return preds

lora_pred_labels = generate_labels(model_lora, test_rows, max_length=LORA_INFER_MAX_LEN)
fallback = weighted_test.argmax(1)
lora_pred_ids = np.array([
    label_map.label2id[lab] if lab in label_map.label2id else int(fallback[i])
    for i, lab in enumerate(lora_pred_labels)
])
results.append(report('lora_gemma_meta', y_test, lora_pred_ids))

## Zero-shot Gemma baseline (no LoRA, same prompts)

In [ ]:
model_lora = model_lora.unload() if hasattr(model_lora, 'unload') else None
torch.cuda.empty_cache()
# Zero-shot sees the full prompt (2048) so it has the same information advantage
# as before training. Use 2048 to give it the best possible context.
zs_pred_labels = generate_labels(base, test_rows, max_length=2048)
zs_pred_ids = np.array([
    label_map.label2id[lab] if lab in label_map.label2id else int(fallback[i])
    for i, lab in enumerate(zs_pred_labels)
])
results.append(report('zero_shot_gemma', y_test, zs_pred_ids))

## Confusion matrix for the LoRA meta-model

In [ ]:
cm = confusion_matrix(y_test, lora_pred_ids, labels=list(range(label_map.num_classes)))
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=label_map.class_names, yticklabels=label_map.class_names,
)
plt.xlabel('Predicted'); plt.ylabel('Gold'); plt.title('LoRA-Gemma meta-model (test)')
plt.tight_layout()
from pathlib import Path
out_dir = Path(REPO_ROOT) / 'Confusion_matrix'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'ensemble_llm_meta.png'
plt.savefig(out_png, dpi=150)
plt.show()
print('saved', out_png)

In [ ]:
summary_path = METRICS_DIR / 'ensemble_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print('saved', summary_path)
pd.DataFrame(results)

In [ ]:
from tm_research.ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()